In [1]:
import xtrack as xt
import sys
helpers_path = f'../' # specify the path to the helper_functions folder
sys.path.insert(0, helpers_path)
from helpers_from_xutil import install_beam_beam_elements, set_BBelem_shift, set_beam_beam_scale

In [2]:
# CHOOSE SEED HERE
seed = 4

In [6]:
line_version = "LCC_106-2-3_z"

# Load corrected line
line = xt.Line.from_json(f'lattices/lattices_with_corrected_imperfections/02_orbit_and_optics_corrected/updated_tolerances/{line_version}_line_optics_corrected_seed{seed}.json')

# just to be safe
line.cycle(name_first_element='rf400', inplace=True)
line.twiss_default['method'] = '6d'
line.configure_radiation(model='mean', model_beamstrahlung=None)
line.compensate_radiation_energy_loss()

Loading line from dict: 100%|██████████| 15545/15545 [00:03<00:00, 4687.89it/s]


Done loading line from dict.           
Compensating energy loss.
Share energy loss among cavities (repeat until energy loss is zero)
Energy loss: 34_787_074.912 eV             
Energy loss: 106_152.107 eV             
Energy loss: 324.292 eV             
Energy loss: 0.991 eV             
Energy loss: -53_917.883 eV             
Energy loss: -246.807 eV             
Energy loss: 163.402 eV             
Energy loss: 1.498 eV             

  - Set delta_taper
  - Restore cavity voltage and frequency. Set cavity lag


In [8]:
tt = line.get_table()
ttips = tt.rows['ip.*']
ttips = ttips.rows[(ttips.name == 'ip.0') | (ttips.name == 'ip.2') | (ttips.name == 'ip.4') | (ttips.name == 'ip.6')] # hardcoded - careful!
ttips

Table: 4 rows, 12 cols
name             s element_type isthick isreplica parent_name parent_type iscollective ...
ip.6       11911.1 Marker         False     False None        None               False
ip.0       34572.3 Marker         False     False None        None               False
ip.2       57233.5 Marker         False     False None        None               False
ip.4       79894.7 Marker         False     False None        None               False

In [9]:
# Define parameters for installing beam-beam
reference_parameters = {
    'normalized_emittance_x': 0.7e-9 * 89236, # tw1.eq_nemitt_x, relativistic gamma factor is 89236
    'normalized_emittance_y': 1.4e-12 * 89236, #tw1.eq_nemitt_y,
    'bunch_length': 16.7e-3, #5.1e-3 when no collisions, 16.7e-3 with collisions
    'bunch_population': 2.02e11}
beam_beam_parameters = {
    'collisions' : True,
    'num_IPs' : 4,
    'half_xing_angle' : 15*1e-3, # half-crossing angle in radians
    'xing_plane' : 0,
    'num_slices' : 251,
    'beamstrahlung_on' : True,
    'binning_mode' : 'unicharge'}

In [10]:
# Install beam-beam elements at the IPs
bb_elem_names = install_beam_beam_elements(line, reference_parameters, beam_beam_parameters, ip_list=list(ttips.name))
print(f"Installed beam-beam elements: {bb_elem_names}")

# Ensure the BB elems are correctly shifted at the IPs (very important since residual orbit is not zero at the IPs!)
tw = line.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
set_BBelem_shift(line, tw)
print(f'Installed {len(bb_elem_names)} beam-beam elements and shifted them correctly.')

# Set the beam-beam strength
bb_strength = 0.80
set_beam_beam_scale(line, bb_strength, ip_list = list(ttips.name))
print(f'Beam-beam elements turned on at {bb_strength}.')

/home/larasievert/miniforge3/envs/fcc-2026/lib/python3.13/site-packages/xtrack/line.py:3330: FutureWarning: Line.insert_element is deprecated. Use Line.insert instead. This deprecation is part of the interface cleanup in view of the 1.0 release.
  warn('Line.insert_element is deprecated. Use Line.insert instead.'


Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Installed beam-beam elements: ['beambeam_ip.6', 'beambeam_ip.0', 'beambeam_ip.2', 'beambeam_ip.4']
Installed 4 beam-beam elements and shifted them correctly.
Beam-beam elements turned on at 0.8.


In [11]:
tw = line.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)